## Suite2p Motion Correction script for functional imaging data 

Manuel Bettencourt 
Date: 25/09/2025
New version: 07/10/2025

__Description:__ 

This code takes in a series of tiff files corresponding to an experiment and outputs a single tiff with all frames of the experiment motion corrected. In the 2 photon computer, the frames are saved (as we set it up) in 10-frame tiffs, which correspond to a single stimulus. Since in each plane, 25 stimuli are presented, there are a total of 250 frames per plane, spread in 25 files. The first thing this code does is to group all the frames corresponding to a plane (group of 25 tiffs) and then merges them to obtain one tiff per plane (with 250 frames). This is done by cells 2 and 3. Then, in cell 5, each plane is motion-corrected using suite 2p motion correction algorithm - this includes rigid and non-rigid registration. In this way, all frames are aligned to each other (_within_ plane alignment). In cell 6, the results of the first alignment are further aligned so that each plane is aligned to the previous one (_between_ plane alignment). To do so, we use the iteratively imporved template (with suite2p's strategy) on the 250 aligned frames of the previous plane as a reference for a rigid registration of the 250 frames of the current plane. In cell 7, we identify bad frames through a simple correlation between the frame and the average of its plane - I found that this deals better with smudged frames than suite2p's automatic badframes identification. Cell 8 pools the bad frames from cell 7 and suite2p's identifiied bad frames and replaces them with NaNs, saving a TIFF file for each plane. cell 4 establishes the key parameters in the algorithm, and cell 1 imports need libraries and establishes the location of the folders for inputs and outputs. The annex was just a simple test to compare using the average of a plane as template against using suite2p's method of doing templates. 

In [1]:
# Cell 1: imports and parameters
import os
import tifffile
from natsort import natsorted
from suite2p import io, registration
from suite2p.registration.register import register_frames
from suite2p.registration.register import compute_reference
from suite2p import default_ops
from suite2p.run_s2p import run_s2p
from suite2p.io.binary import BinaryFile
import numpy as np

# ---------------- USER PARAMETERS ----------------
raw_root   = 'D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional'       # folder with all 10-frame tiffs
merged_dir = 'D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged'   # output folder for merged per-plane movies
save_path1 = 'D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/suite2poutput/withinP'     # Suite2p results folder
save_path2 = 'D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/suite2poutput/betweenP'
nplanes    = 180                       # number of z-planes
frames_per_plane = 250                 # expected frames per plane
files_per_plane = 25                   # number of TIFF files per plane
frames_per_file = 10                   # number of frames in each TIFF
Ly, Lx = 464, 720                      # actual frame size
dtype = np.uint16                      # suite2p binaries are int16
# ------------------------------------------------

os.makedirs(merged_dir, exist_ok=True)
os.makedirs(save_path1, exist_ok=True)
os.makedirs(save_path2, exist_ok=True)

In [2]:
# Cell 2: group TIFFs sequentially by order on disk

# List all TIFF files in the folder and sort lexicographically
all_files = [f for f in os.listdir(raw_root) if f.endswith('.tif')]
all_files = sorted(all_files)  # adjust to natsort if you want natural sort

# Parameters
nplanes = len(all_files) // files_per_plane
print(nplanes)

plane_files = {}

for p in range(nplanes):
    start_idx = p * files_per_plane
    end_idx = start_idx + files_per_plane
    plane_files[p+1] = [os.path.join(raw_root, f) for f in all_files[start_idx:end_idx]]

# Check result
for p in plane_files:
   print(f'Plane {p}: {len(plane_files[p])} files')

180
Plane 1: 25 files
Plane 2: 25 files
Plane 3: 25 files
Plane 4: 25 files
Plane 5: 25 files
Plane 6: 25 files
Plane 7: 25 files
Plane 8: 25 files
Plane 9: 25 files
Plane 10: 25 files
Plane 11: 25 files
Plane 12: 25 files
Plane 13: 25 files
Plane 14: 25 files
Plane 15: 25 files
Plane 16: 25 files
Plane 17: 25 files
Plane 18: 25 files
Plane 19: 25 files
Plane 20: 25 files
Plane 21: 25 files
Plane 22: 25 files
Plane 23: 25 files
Plane 24: 25 files
Plane 25: 25 files
Plane 26: 25 files
Plane 27: 25 files
Plane 28: 25 files
Plane 29: 25 files
Plane 30: 25 files
Plane 31: 25 files
Plane 32: 25 files
Plane 33: 25 files
Plane 34: 25 files
Plane 35: 25 files
Plane 36: 25 files
Plane 37: 25 files
Plane 38: 25 files
Plane 39: 25 files
Plane 40: 25 files
Plane 41: 25 files
Plane 42: 25 files
Plane 43: 25 files
Plane 44: 25 files
Plane 45: 25 files
Plane 46: 25 files
Plane 47: 25 files
Plane 48: 25 files
Plane 49: 25 files
Plane 50: 25 files
Plane 51: 25 files
Plane 52: 25 files
Plane 53: 25 file

In [3]:
# Cell 3: merge 10-frame TIFFs into single 250-frame binary files per plane

# Group files per plane (assuming sequential blocks)
for p in range(nplanes):
    start_idx = p * (frames_per_plane // frames_per_file)
    end_idx = start_idx + (frames_per_plane // frames_per_file)
    plane_files = all_files[start_idx:end_idx]

    # Create binary file for the plane
    bin_filename = os.path.join(merged_dir, f'p{p+1:02d}.bin')
    with open(bin_filename, 'wb') as f_bin:
        for tif_name in plane_files:
            tif_path = os.path.join(raw_root, tif_name)
            stack = tifffile.imread(tif_path)  # shape (10, Ly, Lx)
            stack.astype(np.uint16).tofile(f_bin)

    print(f"Saved plane {p+1} binary: {bin_filename}")

Saved plane 1 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p01.bin
Saved plane 2 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p02.bin
Saved plane 3 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p03.bin
Saved plane 4 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p04.bin
Saved plane 5 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p05.bin
Saved plane 6 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p06.bin
Saved plane 7 binary: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_

In [2]:
# Cell 4: Suite2p options and load files

ops = default_ops()
ops.update({
    'nonrigid': True,            # enable non-rigid registration
    'two_step_registration': True,
    'maxregshift': 0.1,
    'maxregshiftNR': 5,
    'nimg_init': 100,            # frames to build reference
#    'do_bidiphase': True,       # estimate bidirectional offset
#    'bidiphase': 0,             # 0 → use estimated offset
#    'bidi_corrected': True 
    'th_badframes': 0.7   # more stringent: will exclude frames with severe distortion
})

# List all merged bins
bin_files = sorted([f for f in os.listdir(merged_dir) if f.endswith('.bin')])
print(f"Found {len(bin_files)} bin files in {merged_dir}")

#tif_files = sorted([f for f in os.listdir(merged_dir) if f.endswith('.tif')])
#print(f"Found {len(tif_files)} TIFFs in {merged_dir}")

Found 180 bin files in D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged


In [ ]:
# Cell 5: Run Suite2p motion correction (within plane)

output_tif = os.path.join(save_path1, 'motion_corrected_all_planes_within.tif')
reg_dir = os.path.join(save_path1, "registeredBin")  
os.makedirs(reg_dir, exist_ok=True) 

startP=1
endP=180
nplanes=endP-startP+1


# Open TIFF writer for final multi-plane output
with tifffile.TiffWriter(output_tif, bigtiff=True) as tif_writer:

    for p in range(startP, endP+1):
        bin_filename = os.path.join(merged_dir, f'p{p:02d}.bin')
        print(f"\nProcessing plane {p} ...")
        print(bin_filename)

        # Step 1: Create BinaryFile for raw input
        f_raw = io.BinaryFile(Ly=Ly, Lx=Lx, filename=bin_filename, n_frames=frames_per_plane)

        # Step 2: Create BinaryFile for registered output
        reg_filename = os.path.join(reg_dir, f'p{p:02d}_reg.bin')
        f_reg = io.BinaryFile(Ly=Ly, Lx=Lx, filename=reg_filename, n_frames=frames_per_plane)

        # Step 3: run motion correction only
        refImg, rmin, rmax, meanImg, rigid_offsets, \
        nonrigid_offsets, zest, meanImg_chan2, badframes, \
        yrange, xrange = registration.registration_wrapper(
        f_reg, f_raw=f_raw, f_reg_chan2=None, f_raw_chan2=None,
        refImg=None, align_by_chan2=False, ops=ops
        )
        
        # ensure all frames are flushed to disk
        f_reg.file.flush()

        # 🔎 Inspect dropped frames
        badframe_indices = np.where(badframes)[0]
        if len(badframe_indices) > 0:
            print(f"⚠️ {len(badframe_indices)} frames were dropped")
            print("Dropped frame indices:", badframe_indices)
        else:
            print("✅ No frames dropped.")
        
        badframes_filename = os.path.join(reg_dir, f'p{p:02d}_badframes.npy')
        np.save(badframes_filename, badframe_indices)
        #print(f"💾 Saved bad frame indices to {badframes_filename}")


        # Step 4: Save registered frames to combined TIFF
        for i in range(frames_per_plane):
            tif_writer.write(f_reg[i])
        
        #print(f"✅ Saved registered binary: {reg_filename}")
        #print(f"Added plane {p+1} ({frames_per_plane} frames) to {output_tif}")
        


Processing plane 1 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p01.bin
Reference frame, 4.94 sec.
Registered 250/250 in 8.37s
✅ No frames dropped.

Processing plane 2 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p02.bin
Reference frame, 5.42 sec.
Registered 250/250 in 8.08s
✅ No frames dropped.

Processing plane 3 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p03.bin
Reference frame, 4.47 sec.
Registered 250/250 in 7.69s
✅ No frames dropped.

Processing plane 4 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p04.bin
Reference frame, 4.68 sec.
Registered 250/250 in 7.47s
✅ No frames dropped.

Processing plane 5 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6

C:\Users\orger\AppData\Local\Programs\Python\Python39\lib\site-packages\suite2p\registration\register.py:50: RuntimeWarning: invalid value encountered in divide
  dxy = dxy / dxy.mean()


✅ No frames dropped.

Processing plane 33 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p33.bin
Reference frame, 4.77 sec.
Registered 250/250 in 7.50s
⚠️ 1 frames were dropped
Dropped frame indices: [11]

Processing plane 34 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p34.bin
Reference frame, 4.84 sec.
Registered 250/250 in 7.71s
✅ No frames dropped.

Processing plane 35 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p35.bin
Reference frame, 4.69 sec.
Registered 250/250 in 7.78s
✅ No frames dropped.

Processing plane 36 ...
D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/merged\p36.bin
Reference frame, 4.77 sec.
Registered 250/250 in 7.66s
✅ No frames dropped.

Processing plane 37 ...
D:/20251007_hucH2B

In [4]:
# Cell 6: Post-hoc plane-to-plane alignment (sequential) with combined BigTIFF
# Use iteratively aligned template instead of average plane 

# Paths
reg_dir = os.path.join(save_path1, "registeredBin")
aligned_dir = save_path2
os.makedirs(aligned_dir, exist_ok=True)
final_reg_dir = os.path.join(save_path2, "registeredBin")
os.makedirs(final_reg_dir, exist_ok=True)

output_tif = os.path.join(aligned_dir, 'motion_corrected_all_planes_between.tif')
badframes_all_planes_file = os.path.join(aligned_dir, 'badFrames_all_planes.npy')

# Collect registered binaries
reg_files = natsorted([f for f in os.listdir(reg_dir) if f.endswith("_reg.bin")])

aligned_templates = []           # store mean images for sequential reference
total_badframes_all_planes = []  # list of arrays for each plane

ops.update({'nonrigid': False})

# Open BigTIFF writer
with tifffile.TiffWriter(output_tif, bigtiff=True) as tif_writer:

    for i, fname in enumerate(reg_files):
        path_in = os.path.join(reg_dir, fname)
        path_out = os.path.join(final_reg_dir, fname)  # aligned binary output

        print(f"\n🔹 Aligning {fname} ...")

        # Step 1: Load registered binary
        f_raw = io.BinaryFile(Ly=Ly, Lx=Lx, filename=path_in, n_frames=frames_per_plane)
        f_aligned = io.BinaryFile(Ly=Ly, Lx=Lx, filename=path_out, n_frames=frames_per_plane)

        # Step 2: Load bad frames from cell5
        badframes_cell5_file = os.path.join(reg_dir, f'p{i+1:02d}_badframes.npy')
        badframes_cell5 = np.load(badframes_cell5_file) if os.path.exists(badframes_cell5_file) else np.array([], dtype=int)

        # Step 3: Determine reference image
        if i == 0:
            # For first plane, use mean of good frames
            good_idx_plane = np.setdiff1d(np.arange(frames_per_plane), badframes_cell5)
            frames = np.array([f_raw[j] for j in good_idx_plane])
            refImg = compute_reference(frames, ops)
        else:
            # Reference is previous plane template
            refImg = aligned_templates[-1]

        # Step 4: Align all frames in this plane to reference
        refImg, rmin, rmax, meanImg, rigid_offsets, \
        nonrigid_offsets, zest, meanImg_chan2, badframes_cell6, \
        yrange, xrange = registration.registration_wrapper(
            f_aligned,
            f_raw=f_raw,
            f_reg_chan2=None,
            f_raw_chan2=None,
            refImg=refImg,
            align_by_chan2=False,
            ops=ops
        )

        # Step 5: Combine bad frames from cell5 + cell6
        total_badframes_plane = np.union1d(badframes_cell5, np.where(badframes_cell6)[0])
        total_badframes_all_planes.append(total_badframes_plane)

        # Step 6: Save all aligned frames to BigTIFF (including bad frames)
        for j in range(frames_per_plane):
            tif_writer.write(f_aligned[j])

        # Step 7: Save mean of good frames for next sequential reference
        good_idx_plane = np.setdiff1d(np.arange(frames_per_plane), total_badframes_plane)
        if len(good_idx_plane) > 0:
            frames = np.array([f_aligned[j] for j in good_idx_plane])
            template = compute_reference(frames, ops)
        else:
            template = aligned_templates[-1] if aligned_templates else np.zeros((Ly, Lx), dtype=np.float32)
        aligned_templates.append(template)

        # Optional: report dropped frames
        print(f"⚠️ Plane {i+1}: {len(total_badframes_plane)} total bad frames")
        if len(total_badframes_plane) > 0:
            print("Dropped frame indices:", total_badframes_plane)

        # Flush BinaryFile to disk
        f_aligned.file.flush()

# Step 8: Save cumulative bad frames for all planes
badframes_dict = {f"plane_{i+1:02d}": total_badframes_all_planes[i] for i in range(len(total_badframes_all_planes))}
np.save(badframes_all_planes_file, badframes_dict, allow_pickle=True)
print(f"\n✅ All planes aligned and saved to BigTIFF: {output_tif}")
#print(f"💾 Cumulative bad frames saved to: {badframes_all_planes_file}")


🔹 Aligning p01_reg.bin ...
Registered 250/250 in 1.46s
⚠️ Plane 1: 0 total bad frames

🔹 Aligning p02_reg.bin ...
Registered 250/250 in 2.31s
⚠️ Plane 2: 0 total bad frames

🔹 Aligning p03_reg.bin ...
Registered 250/250 in 2.42s
⚠️ Plane 3: 0 total bad frames

🔹 Aligning p04_reg.bin ...
Registered 250/250 in 2.58s
⚠️ Plane 4: 0 total bad frames

🔹 Aligning p05_reg.bin ...
Registered 250/250 in 2.51s
⚠️ Plane 5: 0 total bad frames

🔹 Aligning p06_reg.bin ...
Registered 250/250 in 2.40s
⚠️ Plane 6: 0 total bad frames

🔹 Aligning p07_reg.bin ...
Registered 250/250 in 2.42s
⚠️ Plane 7: 0 total bad frames

🔹 Aligning p08_reg.bin ...
Registered 250/250 in 2.35s
⚠️ Plane 8: 0 total bad frames

🔹 Aligning p09_reg.bin ...
Registered 250/250 in 2.53s
⚠️ Plane 9: 0 total bad frames

🔹 Aligning p10_reg.bin ...
Registered 250/250 in 2.30s
⚠️ Plane 10: 0 total bad frames

🔹 Aligning p11_reg.bin ...
Registered 250/250 in 2.35s
⚠️ Plane 11: 1 total bad frames
Dropped frame indices: [90]

🔹 Aligning p

In [2]:
#Cell 7: Find out bad frames through correlation

# Paths
reg_dir = os.path.join(save_path2, "registeredBin")  # folder with registered .bin files
badframes_outfile = os.path.join(save_path2, "badFrames_corr_all_planes.npy")  # final output

# Parameters
threshold = 2.5  # number of standard deviations below the mean 

# Get all registered binaries and sort naturally
reg_files = natsorted([f for f in os.listdir(reg_dir) if f.endswith("_reg.bin")])

# Dictionary to store bad frames per plane
badframes_dict = {}
corr_dict = {}
# Loop over planes
for i, fname in enumerate(reg_files):
    path_in = os.path.join(reg_dir, fname)
    f_raw = BinaryFile(Ly=Ly, Lx=Lx, filename=path_in, n_frames=frames_per_plane)
    
    # Compute plane average
    avg_img = np.mean([f_raw[j] for j in range(frames_per_plane)], axis=0)
    
    # Compute correlation of each frame with average
    corr_list=[]
    badframes_plane = []
    for j in range(frames_per_plane):
        frame = f_raw[j]
        corr = np.corrcoef(frame.flatten(), avg_img.flatten())[0, 1]
        corr_list.append(corr)
    
    
    for k in range(frames_per_plane):
        corr = corr_list[k]
        mean_corr = np.mean(corr_list)
        sd_corr = np.std(corr_list)
        if corr < mean_corr-threshold*sd_corr:
            badframes_plane.append(k)
    
    badframes_dict[fname] = np.array(badframes_plane)
    corr_dict[fname] = np.array(corr_list)
    print(f"Plane {fname}: {len(badframes_plane)}/{frames_per_plane} bad frames detected")

# Save dictionary
np.save(badframes_outfile, badframes_dict)
print(f"\n💾 Saved correlation-based bad frames for all planes to {badframes_outfile}")

Plane p01_reg.bin: 4/250 bad frames detected
Plane p02_reg.bin: 8/250 bad frames detected
Plane p03_reg.bin: 4/250 bad frames detected
Plane p04_reg.bin: 1/250 bad frames detected
Plane p05_reg.bin: 6/250 bad frames detected
Plane p06_reg.bin: 6/250 bad frames detected
Plane p07_reg.bin: 4/250 bad frames detected
Plane p08_reg.bin: 4/250 bad frames detected
Plane p09_reg.bin: 6/250 bad frames detected
Plane p10_reg.bin: 5/250 bad frames detected
Plane p11_reg.bin: 1/250 bad frames detected
Plane p12_reg.bin: 4/250 bad frames detected
Plane p13_reg.bin: 3/250 bad frames detected
Plane p14_reg.bin: 2/250 bad frames detected
Plane p15_reg.bin: 4/250 bad frames detected
Plane p16_reg.bin: 3/250 bad frames detected
Plane p17_reg.bin: 3/250 bad frames detected
Plane p18_reg.bin: 2/250 bad frames detected
Plane p19_reg.bin: 3/250 bad frames detected
Plane p20_reg.bin: 3/250 bad frames detected
Plane p21_reg.bin: 3/250 bad frames detected
Plane p22_reg.bin: 6/250 bad frames detected
Plane p23_

In [6]:
# Cell 8: Save per-plane TIFFs where bad frames are replaced with NaNs 


# Paths
aligned_dir = save_path2
final_reg_dir = os.path.join(aligned_dir, "registeredBin")
total_badframes_path = os.path.join(aligned_dir, 'badFrames_all_planes.npy')
corr_badframes_path = os.path.join(aligned_dir, 'badFrames_corr_all_planes.npy')
output_dir = os.path.join(aligned_dir, "clean_planes_nan")
os.makedirs(output_dir, exist_ok=True)

# Load both bad frame dictionaries
total_badframes_dict = np.load(total_badframes_path, allow_pickle=True).item()
corr_badframes_dict = np.load(corr_badframes_path, allow_pickle=True).item()

# Get registered binary files in correct numeric order
reg_files = natsorted([f for f in os.listdir(final_reg_dir) if f.endswith("_reg.bin")])

print(f"Found {len(reg_files)} registered planes.")
print(f"Saving cleaned TIFFs (NaN in bad frames) to: {output_dir}")

for i, fname in enumerate(reg_files):
    path_in = os.path.join(final_reg_dir, fname)
    print(f"\nProcessing plane {i+1}/{len(reg_files)}: {fname}")

    # Load registered binary
    f_raw = io.BinaryFile(Ly=Ly, Lx=Lx, filename=path_in, n_frames=frames_per_plane)

    # --- Combine bad frames from both sources ---
    plane_key = f"plane_{i+1:02d}"
    bad1 = total_badframes_dict.get(plane_key, total_badframes_dict.get(fname, np.array([], dtype=int)))
    bad2 = corr_badframes_dict.get(fname, np.array([], dtype=int))
    merged_bad = np.unique(np.concatenate([bad1, bad2])) if bad1.size or bad2.size else np.array([], dtype=int)

    # --- Save this plane's TIFF ---
    output_tif_path = os.path.join(output_dir, fname.replace("_reg.bin", "_nan.tif"))
    with tifffile.TiffWriter(output_tif_path, bigtiff=True) as tif_writer:
        for j in range(frames_per_plane):
            if j in merged_bad:
                # Replace bad frames with NaNs (same shape)
                frame = np.full((Ly, Lx), np.nan, dtype=np.float32)
            else:
                frame = f_raw[j].astype(np.float32)
            tif_writer.write(frame)

    print(f"✅ Plane {i+1}: wrote {frames_per_plane} frames ({len(merged_bad)} replaced with NaN) → {os.path.basename(output_tif_path)}")

print("\n💾 All per-plane TIFFs saved with NaN-filled bad frames.")
print(f"Output folder: {output_dir}")

Found 180 registered planes.
Saving cleaned TIFFs (NaN in bad frames) to: D:/20251007_hucH2BGCaMP6s_vglut2aDsRed/f1_6dpf_2photon/20251007_hucH2BGCaMP6s_vglut2aDsRed_6dpf_f1_functional/suite2poutput/betweenP\clean_planes_nan

Processing plane 1/180: p01_reg.bin
✅ Plane 1: wrote 250 frames (3 replaced with NaN) → p01_nan.tif

Processing plane 2/180: p02_reg.bin
✅ Plane 2: wrote 250 frames (7 replaced with NaN) → p02_nan.tif

Processing plane 3/180: p03_reg.bin
✅ Plane 3: wrote 250 frames (3 replaced with NaN) → p03_nan.tif

Processing plane 4/180: p04_reg.bin
✅ Plane 4: wrote 250 frames (0 replaced with NaN) → p04_nan.tif

Processing plane 5/180: p05_reg.bin
✅ Plane 5: wrote 250 frames (4 replaced with NaN) → p05_nan.tif

Processing plane 6/180: p06_reg.bin
✅ Plane 6: wrote 250 frames (3 replaced with NaN) → p06_nan.tif

Processing plane 7/180: p07_reg.bin
✅ Plane 7: wrote 250 frames (4 replaced with NaN) → p07_nan.tif

Processing plane 8/180: p08_reg.bin
✅ Plane 8: wrote 250 frames (1 r

In [19]:
#Annex 1
#Comparing using plain average vs using template generated by suite2p iterative alignment procedure 

import matplotlib.pyplot as plt
from suite2p.registration.register import compute_reference

reg_dir = os.path.join(save_path1, "registeredBin")
reg_files = natsorted([f for f in os.listdir(reg_dir) if f.endswith("_reg.bin")])
Plane1 = reg_files[0]
print(Plane1)
badframes_plane1_file = os.path.join(reg_dir, 'p12_badframes.npy')
badframes_plane1 = np.load(badframes_plane1_file) 
print(badframes_plane1)
f_raw = io.BinaryFile(Ly=Ly, Lx=Lx, filename=os.path.join(reg_dir, 'p12_reg.bin'), n_frames=250)
good_idx_plane = np.setdiff1d(np.arange(250), badframes_plane1)
frames = np.array([f_raw[j] for j in good_idx_plane])
refImg_mean = np.mean(frames, axis=0)
ops = default_ops()
ops.update({
    'nonrigid': False,            
    'th_badframes': 0.7})
refImg_template = compute_reference(frames, ops)

# --- Save as TIFFs for Fiji ---
template_test_dir = os.path.join(save_path2, "template_test")
os.makedirs(template_test_dir, exist_ok=True)

tif_mean_path = os.path.join(template_test_dir, "plane1_mean_goodframes.tif")
tif_template_path = os.path.join(template_test_dir, "plane1_suite2p_template.tif")

tifffile.imwrite(tif_mean_path, refImg_mean.astype(np.float32))
tifffile.imwrite(tif_template_path, refImg_template.astype(np.float32))


p12_reg.bin
[107]


In [13]:
f_raw = io.BinaryFile(Ly=Ly, Lx=Lx, filename=os.path.join(reg_dir, 'p152_reg.bin'), n_frames=250)

ValueError: cannot mmap an empty file